# Forecast: a daily metric, the next weeks

A forecast of one number a day (revenue, orders, sign-ups) for the next `HORIZON` days, from its history: FLAML's AutoML tries time-series models (statsmodels' ARIMA, SARIMAX and Holt-Winters, and tree models on the calendar), judged on the last `HORIZON` days it's kept back; the best is logged and registered in the project's own MLflow workspace.

**To use it:** save a copy (File → Save Notebook As…), then change the parameters in the next cell: the project, the query that gives a value per day, its two columns, the horizon and the model's name. The example is the guide's `retail` project's revenue; run top to bottom. Data comes through the platform's SQL, as you, at most 100,000 rows: a row per day, aggregated in SQL (about two and a half years).

In [ ]:
import os

PROJECT = os.environ.get("STORSCALE_PROJECT", "retail")   # a scheduled run: the project it runs for                 # the project: its tables, its MLflow workspace, its Models tab
SERIES = """
SELECT ordered_on AS day, CAST(sum(amount) AS double) AS revenue
FROM uploads_orders
WHERE NOT test AND NOT refunded
GROUP BY ordered_on
"""                                # a value per day
DATE = "day"                       # the day column
VALUE = "revenue"                  # the value column
HORIZON = 28                       # days to forecast (and to hold back to judge the models)
MODEL = "revenue_forecast"         # the registered model's name
EXPERIMENT = "forecast"            # the MLflow experiment its runs go to
TIME_BUDGET = 60                   # seconds FLAML may spend trying models

## The data

A value for every day: a day with no rows counts as 0 (right for revenue or orders; for a level like stock or a price, fill forward instead: `.ffill()`).

In [ ]:
import buckets_lake
import pandas as pd

raw = buckets_lake.sql(PROJECT, SERIES)            # as you, under the project's rules
series = (raw.assign(**{DATE: pd.to_datetime(raw[DATE])}).groupby(DATE)[VALUE].sum()
          .asfreq("D", fill_value=0.0).astype("float64"))
data = series.rename_axis(DATE).reset_index()
assert len(data) >= 3 * HORIZON, f"{len(data)} days: a forecast of {HORIZON} wants three times that at least"
print(f"{len(data)} days, {series.index.min():%Y-%m-%d} to {series.index.max():%Y-%m-%d}")
series.describe()

## Train, log and register

FLAML judges each model on the last `HORIZON` days by mean absolute error, then learns the best again from every day. The same days' error of a naive forecast (each day as the same weekday a week before) says whether the model earns its keep.

In [ ]:
from flaml import AutoML

mlflow = buckets_lake.mlflow(PROJECT)        # the project's own MLflow workspace, as you
mlflow.set_experiment(EXPERIMENT)
future = pd.DataFrame({DATE: pd.date_range(series.index[-1] + pd.Timedelta(days=1), periods=HORIZON, freq="D")})
with mlflow.start_run():                     # stamped with you and the catalog's commit
    automl = AutoML()
    automl.fit(dataframe=data, label=VALUE, task="ts_forecast", period=HORIZON, metric="mae", time_budget=TIME_BUDGET,
               estimator_list=["arima", "sarimax", "holt-winters", "lgbm", "rf", "xgb_limitdepth", "extra_tree"],
               eval_method="holdout", seed=0, verbose=0, mlflow_logging=False)
    naive = (series - series.shift(7)).iloc[-HORIZON:].abs().mean()
    forecast = future.assign(**{VALUE: list(automl.predict(future))})
    mlflow.log_params({"query": SERIES.strip(), "days": len(data), "horizon": HORIZON, "estimator": automl.best_estimator,
                       **{f"best_{k}": v for k, v in automl.best_config.items()}})
    mlflow.log_metrics({"mae": automl.best_loss, "mae_naive": naive})
    mlflow.log_text(forecast.to_csv(index=False), "forecast.csv")
    # The whole AutoML object: it turns days into a forecast. It loads where FLAML is (notebooks).
    mlflow.sklearn.log_model(automl, name="model", registered_model_name=MODEL, serialization_format="cloudpickle",
                             input_example=future.head())
print(f"{automl.best_estimator}: mean absolute error {automl.best_loss:,.2f} a day on the last {HORIZON} days; naive {naive:,.2f}")

In [ ]:
forecast   # the next HORIZON days

## Next

1. **Its card, and an approval.** On the project's **Models** tab, `revenue_forecast` has a new version, registered by you. **Write its card** (what it forecasts, the history it learned from, its error against the naive forecast, its limits), and have another editor of the project approve the version: nobody approves their own. The alias `approved` then names it.
2. **Use it.** The forecast starts the day after its history ends: load the approved version in a notebook and give it the days that follow (the run that trained it also kept its forecast, `forecast.csv`, among its artifacts):

   ```python
   import buckets_lake, pandas as pd
   mlflow = buckets_lake.mlflow("retail")
   model = mlflow.pyfunc.load_model("models:/revenue_forecast@approved")
   days = pd.DataFrame({"day": pd.date_range("<the day after the history>", periods=28, freq="D")})
   forecast = days.assign(revenue=list(model.predict(days)))
   buckets_lake.s3().put_object(Bucket="retail", Key="landing/forecasts/revenue.csv", Body=forecast.to_csv(index=False).encode())
   ```

   In the project's bucket, the landing area's file can be imported as a table (the **Data** tab) for dashboards to draw beside the actuals.
3. **Retrain** as the days come in (weekly, say): run this notebook again for a forecast from the latest day, and have the new version approved. A model that keeps losing to `mae_naive` isn't worth its card.